> **Quick start — click *Run All* — no setup needed.**
> Cached PNG figures are displayed by default (`RERUN = False`).
> Set `RERUN = True` and re-run to regenerate figures from the source scripts.

In [ ]:
import subprocess, sys, pathlib
from IPython.display import Image, display

# locate repository root by searching upward for lunar/__init__.py
_here = pathlib.Path.cwd()
REPO = None
for _p in [_here, *_here.parents]:
    if (_p / "lunar" / "__init__.py").exists():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError("Cannot find REPO root — run from inside Lunar-V2/")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

FIGS    = REPO / "output" / "figures"
SCRIPTS = REPO / "scripts" / "phase2"


def show_figure(name: str, caption: str = "") -> None:
    """Display a cached PNG from output/figures/."""
    path = FIGS / name
    if not path.exists():
        print(f"[warn] figure not found: {path}")
        return
    display(Image(str(path), width=900))
    if caption:
        from IPython.display import Markdown
        display(Markdown(f"*{caption}*"))


def run_script(name: str) -> None:
    """Run a Phase-2 script via subprocess."""
    script = SCRIPTS / name
    print(f"Running: {script}")
    result = subprocess.run(
        [sys.executable, str(script)],
        cwd=str(REPO),
        capture_output=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Script exited with code {result.returncode}: {script}")

print(f"REPO    = {REPO}")
print(f"FIGS    = {FIGS}  (exists={FIGS.exists()})")
print(f"SCRIPTS = {SCRIPTS}  (exists={SCRIPTS.exists()})")


# Phase 2 (Craters) — Bowl-crater illumination model

Mirrors the upstream `Craters/` directory from Martinez & Siegler (2021). Covers `crater_floor_insolation()` (port of `insolationcrater.m`) and the existing LOLA horizon tracer in `lunar/illumination.py`.

## §1 — `crater_floor_insolation()` — port of upstream `insolationcrater.m`

The floor-averaged absorbed solar flux for a bowl crater:

$$Q_s = \frac{S \cdot 4\varepsilon(1-A)}{D^2}\left(1 + \frac{A}{\varepsilon}\right)\max(0,\cos\theta)$$

where *S* = 1361 W m⁻², *A* = albedo, *ε* = emissivity, *D* = normalised diameter-to-depth ratio, *θ* = solar incidence angle at the crater floor.

> **Note:** Valid for generic bowl craters. For **Shoemaker PSR** specifically the upstream uses a precomputed ray-traced time series — see `03_psr_shoemaker.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lunar.illumination import crater_floor_insolation

t = np.linspace(0, 2.55024e6, 2000)  # one lunation in seconds
Q = crater_floor_insolation(t, latitude_deg=-70, diameter_norm=3.0)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t / 86400, Q)
ax.set_xlabel('Time (days)')
ax.set_ylabel('Absorbed flux (W/m²)')
ax.set_title('Bowl-crater floor insolation: lat=−70°, D_norm=3.0')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## §2 — Horizon tracer (`lunar/illumination.py`)

Implements the **Mazarico 2011** (*Icarus*) method: for each azimuth bin, the horizon elevation angle is determined by scanning radially outward in the DEM. Shadow is cast whenever the solar elevation angle falls below the local horizon angle.

`compute_horizon()` produces horizon-angle arrays equivalent to those underlying `shoemakerIllumination.mat` for arbitrary new sites. For a full demo see `notebooks/phase2_illumination/04_illumination_shadows.ipynb` (legacy dir).

In [ ]:
import inspect
from lunar.illumination import compute_horizon, synthetic_crater_dem

print('compute_horizon signature:')
print(inspect.signature(compute_horizon))
print()
print(compute_horizon.__doc__[:400] if compute_horizon.__doc__ else '(no docstring)')